# Random Forest

Ensemble method that builds multiple decision trees using bagging (bootstrap aggregation) and combines their predictions. Handles non-linear relationships, provides feature importance, and is resistant to overfitting with sufficient trees.

**When to Use:**
- Non-linear relationships between features and target
- Feature importance ranking is needed
- No strong assumptions about data distribution required
- Mixed feature types (numeric + categorical)
- Robust baseline model for both classification and regression

**Key Assumptions / Considerations:**
- Independent observations (bootstrap sampling assumes this)
- More trees generally improve performance but with diminishing returns
- Can overfit on noisy datasets if trees are too deep
- Feature importance can be biased toward high-cardinality features
- OOB (Out-of-Bag) error provides a built-in validation estimate

**References:**
- [sklearn RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [sklearn RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [sklearn permutation_importance](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html)

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score, 
    roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# test data 

np.random.seed(100)
n_samples = 8000

# numeric
age = np.random.randint(18, 70, n_samples)
income = np.random.normal(50000, 15000, n_samples)
num_transactions = np.random.poisson(10, n_samples)

# categorical
gender = np.random.choice(['Male', 'Female'], n_samples)
region = np.random.choice(['North', 'South', 'East', 'West'], n_samples)
product_type = np.random.choice(['A', 'B', 'C'], n_samples)

# continuous target
target_continuous = 0.05 * age + 0.0005 * income + 0.3 * num_transactions + np.random.normal(0, 2, n_samples)

# binary classification target
target_binary = (target_continuous > np.median(target_continuous)).astype(int)


df = pd.DataFrame({
    'age': age,
    'income': income,
    'num_transactions': num_transactions,
    'gender': gender,
    'region': region,
    'product_type': product_type,
    'target': target_continuous  # switch to target_binary for classification 
})

In [ ]:
# Load Data

#df = pd.read_csv("data.csv")

X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# Distribution

plt.figure(figsize=(8, 5))
sns.histplot(y, kde=True, bins=30)
plt.title("Target Variable Distribution")
plt.show()

print("Skewness:", round(y.skew(),4))
print("Kurtosis:", round(y.kurt(), 4))

In [ ]:
# Train/Test Split

try:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    # if y is continuous
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

In [ ]:
# Preprocessing

numeric_features = ['age', 'income', 'num_transactions']
categorical_features = ['gender', 'region', 'product_type']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
# Parameter Grids

models = {
    "RF_Regressor": (RandomForestRegressor(random_state=42, n_jobs=-1, oob_score=True), {
        "clf__n_estimators": [100, 300, 500],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_split": [2, 5, 10],
        "clf__max_features": ["sqrt", "log2", None],
    }),
    "RF_Classifier": (RandomForestClassifier(random_state=42, n_jobs=-1, oob_score=True), {
        "clf__n_estimators": [100, 300, 500],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_split": [2, 5, 10],
        "clf__max_features": ["sqrt", "log2"],
    }),
}

# Cross Validation

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1) if len(y.unique()) < 10 else KFold(n_splits=5, shuffle=True, random_state=1)

In [ ]:
# Training

def train_models(X_train, y_train, X_test, y_test):

    results = []
    fitted_pipelines = []

    for name, (model, params) in models.items():
        print(f"\n\U0001f539 Training {name}...")

        is_classifier = isinstance(model, RandomForestClassifier)

        pipe = Pipeline([("prep", preprocessor), ("clf", model)])

        scoring = "roc_auc" if is_classifier else "r2"

        grid = GridSearchCV(pipe, param_grid=params, cv=cv, n_jobs=-1, scoring=scoring, return_train_score=False)

        try:
            y_train_use = y_train if not is_classifier else (y_train > np.median(y_train)).astype(int)
            y_test_use = y_test if not is_classifier else (y_test > np.median(y_test)).astype(int)

            grid.fit(X_train, y_train_use)
            best_model = grid.best_estimator_
            fitted_pipelines.append((name, best_model))
            y_pred = best_model.predict(X_test)

            if is_classifier:
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "Accuracy": accuracy_score(y_test_use, y_pred),
                    "Recall": recall_score(y_test_use, y_pred),
                    "Precision": precision_score(y_test_use, y_pred),
                    "F1 Score": f1_score(y_test_use, y_pred),
                    "ROC-AUC": roc_auc_score(y_test_use, best_model.predict_proba(X_test)[:, 1]),
                    "OOB Score": best_model.named_steps["clf"].oob_score_,
                }
            else:
                n = len(y_test_use)
                p = X_train.shape[1]
                rss = np.sum((y_test_use - y_pred) ** 2)
                mse = mean_squared_error(y_test_use, y_pred)
                aic = n * np.log(rss / n) + 2 * p
                bic = n * np.log(rss / n) + p * np.log(n)
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "RMSE": np.sqrt(mse),
                    "MAE": mean_absolute_error(y_test_use, y_pred),
                    "R\u00b2 Score": r2_score(y_test_use, y_pred),
                    "AIC": aic,
                    "BIC": bic,
                    "OOB Score": best_model.named_steps["clf"].oob_score_,
                }

            results.append(metrics)
        except Exception as e:
            print(f"\u26a0\ufe0f Skipping {name} due to error: {e}")
            continue

    return results, fitted_pipelines

In [ ]:
results, pipelines = train_models(X_train_full, y_train_full, X_test, y_test)

In [ ]:
# Results Summary

results_df = pd.DataFrame(results)
print("\n\u2705 Model Evaluation Summary:")
results_df

In [ ]:
# Best Model by R\u00b2 (regression) or ROC-AUC (classification)
regression_results = [r for r in results if 'R\u00b2 Score' in r]
if regression_results:
    best_model_name = max(regression_results, key=lambda x: x['R\u00b2 Score'])['Model']
else:
    best_model_name = max(results, key=lambda x: x.get('ROC-AUC', 0))['Model']

best_model_pipeline = [p for n, p in pipelines if n == best_model_name][0]
print(f"\n\U0001f3c6 Best Model: {best_model_name}")

In [ ]:
# Feature Importance (Mean Decrease Impurity)

for name, pipe in pipelines:
    print(f"\nModel: {name}")
    
    rf_model = pipe.named_steps["clf"]
    importances = rf_model.feature_importances_
    
    prep = pipe.named_steps["prep"]
    feature_names = prep.get_feature_names_out()
    
    df_imp = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_imp, x="importance", y="feature")
    plt.title(f"{name} - MDI Feature Importance")
    plt.tight_layout()
    plt.show()
    
    print(df_imp)

In [ ]:
# Permutation Importance (on test set)

for name, pipe in pipelines:
    is_classifier = "Classifier" in name
    y_test_use = y_test if not is_classifier else (y_test > np.median(y_test)).astype(int)
    
    scoring = "accuracy" if is_classifier else "r2"
    perm_imp = permutation_importance(pipe, X_test, y_test_use, n_repeats=10, random_state=42, scoring=scoring)
    
    prep = pipe.named_steps["prep"]
    feature_names = prep.get_feature_names_out()
    
    df_perm = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": perm_imp.importances_mean,
        "importance_std": perm_imp.importances_std
    }).sort_values("importance_mean", ascending=False)
    
    plt.figure(figsize=(10, 5))
    plt.barh(df_perm["feature"], df_perm["importance_mean"], xerr=df_perm["importance_std"])
    plt.title(f"{name} - Permutation Importance")
    plt.xlabel("Mean Importance")
    plt.tight_layout()
    plt.show()

In [ ]:
# Regression Diagnostics

def regression_diagnostics(model, X, y, name="Model"):
    y_pred = model.predict(X)
    residuals = y - y_pred
    fitted = y_pred

    plt.figure(figsize=(12, 10))
    
    # Residuals vs Fitted
    plt.subplot(2,2,1)
    sns.scatterplot(x=fitted, y=residuals)
    plt.axhline(0, color='r', linestyle='--')
    plt.xlabel("Fitted values")
    plt.ylabel("Residuals")
    plt.title(f"{name} - Residuals vs Fitted")
    
    # Histogram of residuals
    plt.subplot(2,2,2)
    sns.histplot(residuals, kde=True, bins=30)
    plt.title(f"{name} - Residual Distribution")
    
    # Scale-Location
    plt.subplot(2,2,3)
    sns.scatterplot(x=fitted, y=np.sqrt(np.abs(residuals)))
    plt.xlabel("Fitted values")
    plt.ylabel("Sqrt(|Residuals|)")
    plt.title(f"{name} - Scale-Location")
    
    # Predicted vs Actual
    plt.subplot(2,2,4)
    sns.scatterplot(x=y, y=fitted)
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{name} - Predicted vs Actual")
    
    plt.tight_layout()
    plt.show()

# Run on regression models
for name, pipe in pipelines:
    if "Regressor" in name:
        regression_diagnostics(pipe, X_train_full, y_train_full, f"{name} (Train)")
        regression_diagnostics(pipe, X_test, y_test, f"{name} (Test)")

In [ ]:
# Classification Diagnostics

def classification_diagnostics(model, X, y_true, name="Model"):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].set_title(f"{name} - Confusion Matrix")
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    axes[1].plot([0, 1], [0, 1], 'r--')
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"{name} - ROC Curve")
    axes[1].legend()
    
    # Precision-Recall Curve
    prec, rec, _ = precision_recall_curve(y_true, y_proba)
    axes[2].plot(rec, prec)
    axes[2].set_xlabel("Recall")
    axes[2].set_ylabel("Precision")
    axes[2].set_title(f"{name} - Precision-Recall Curve")
    
    plt.tight_layout()
    plt.show()
    
    print(classification_report(y_true, y_pred))

# Run on classifier models
for name, pipe in pipelines:
    if "Classifier" in name:
        y_test_binary = (y_test > np.median(y_test)).astype(int)
        classification_diagnostics(pipe, X_test, y_test_binary, name)

In [ ]:
# Profile Plots

def plot_profiles(best_pipeline, X, y_true):
    y_pred = best_pipeline.predict(X)
    
    n_cols = 3
    n_features = X.shape[1]
    n_rows = math.ceil(n_features / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()  

    for i, col in enumerate(X.columns):
        temp = pd.DataFrame({
            col: X[col],
            "y_true": y_true,
            "y_pred": y_pred
        })

        grouped = temp.groupby(col).agg(
            count=("y_true", "size"),
            avg_true=("y_true", "mean"),
            avg_pred=("y_pred", "mean")
        ).reset_index().sort_values(col)

        ax1 = axes[i]
        ax1.bar(grouped[col], grouped["count"], alpha=0.3)
        ax1.set_xlabel(col)
        ax1.set_ylabel("Population")

        ax2 = ax1.twinx()
        ax2.plot(grouped[col], grouped["avg_true"], marker="o", label="Actual Target")
        ax2.plot(grouped[col], grouped["avg_pred"], marker="o", linestyle="--", label="Predicted Target")
        ax2.set_ylabel("Target Value")

        ax1.set_title(col)
        ax2.legend(loc="upper right")

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

plot_profiles(best_model_pipeline, X_train_full, y_train_full)

In [ ]:
# future work:
#   hyperparameter tuning with RandomizedSearchCV for faster search on large grids
#   feature engineering: interactions, polynomial features
#   compare with ExtraTrees (ExtraTreesRegressor/Classifier) for speed
#   partial dependence plots (sklearn.inspection.PartialDependenceDisplay)
#   SHAP values for more detailed feature interaction analysis